# 02 — Chạy Genetic Algorithm và xuất thời khóa biểu

Notebook chạy luồng production **GA + Repair + Soft Local Search**, xuất Excel,
JSON và biểu đồ hội tụ. Kết quả được đồng bộ sang Google Drive để notebook demo
sử dụng đúng lịch vừa tạo.

In [ ]:
#@title Clone dự án từ GitHub và cài môi trường Colab
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
from pathlib import Path
import shutil
import subprocess
import sys

THU_MUC_COLAB_DRIVE = Path("/content/drive/MyDrive/Genetic_ALO_Colab")
THU_MUC_COLAB_DRIVE.mkdir(parents=True, exist_ok=True)
THU_MUC_DU_AN = Path("/content/genetic-alo")
THU_MUC_KET_QUA_DRIVE = THU_MUC_COLAB_DRIVE / "latest_outputs"
REPOSITORY_URL = "https://github.com/duktrung05/genetic-alo.git"

shutil.rmtree(THU_MUC_DU_AN, ignore_errors=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "main",
        REPOSITORY_URL, str(THU_MUC_DU_AN),
    ],
    check=True,
)

# Khôi phục output mới nhất từ Drive nếu notebook trước đã tạo kết quả.
if THU_MUC_KET_QUA_DRIVE.is_dir():
    shutil.copytree(
        THU_MUC_KET_QUA_DRIVE,
        THU_MUC_DU_AN / "outputs",
        dirs_exist_ok=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "-r",
        str(THU_MUC_DU_AN / "requirements.txt"),
    ],
    check=True,
)

os.chdir(THU_MUC_DU_AN)
if str(THU_MUC_DU_AN) not in sys.path:
    sys.path.insert(0, str(THU_MUC_DU_AN))

def dong_bo_ket_qua() -> Path:
    """Copy all current outputs to Drive so another notebook can reuse them."""
    THU_MUC_KET_QUA_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        THU_MUC_DU_AN / "outputs",
        THU_MUC_KET_QUA_DRIVE,
        dirs_exist_ok=True,
    )
    return THU_MUC_KET_QUA_DRIVE

print(f"✅ Đã clone dự án tại: {THU_MUC_DU_AN}")
print(f"✅ Python: {sys.version.split()[0]}")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=THU_MUC_DU_AN, check=True)
print("✅ Dataset Excel nằm trong data/instances.")

In [ ]:
#@title Cấu hình và chạy thuật toán
TEN_DATASET = "easy" #@param ["easy", "medium"]
SEED = 42 #@param {type:"integer"}
KICH_THUOC_QUAN_THE = 60 #@param {type:"integer"}
NGAN_SACH_DANH_GIA = 1000 #@param {type:"integer"}
DUNG_SOFT_LOCAL_SEARCH = True #@param {type:"boolean"}
HO_SO_TRONG_SO = "balanced" #@param ["student_centric", "balanced", "resource_centric"]

duong_dan_input = THU_MUC_DU_AN / f"data/instances/instance_{TEN_DATASET}.xlsx"
thu_muc_ket_qua = THU_MUC_DU_AN / "outputs/production"
thu_muc_ket_qua.mkdir(parents=True, exist_ok=True)
duong_dan_output = thu_muc_ket_qua / "best_timetable.xlsx"

lenh_chay = [
    sys.executable,
    "main.py",
    "--input", str(duong_dan_input),
    "--output", str(duong_dan_output),
    "--seed", str(SEED),
    "--population-size", str(KICH_THUOC_QUAN_THE),
    "--search-evaluation-budget", str(NGAN_SACH_DANH_GIA),
    "--weight-profile", HO_SO_TRONG_SO,
]
if DUNG_SOFT_LOCAL_SEARCH:
    lenh_chay.append("--soft-local-search")

print("Lệnh thực thi:", " ".join(lenh_chay))
subprocess.run(lenh_chay, cwd=THU_MUC_DU_AN, check=True)
vi_tri_drive = dong_bo_ket_qua()
print(f"✅ File thời khóa biểu: {duong_dan_output}")
print(f"✅ Đã đồng bộ output sang: {vi_tri_drive}")

In [ ]:
#@title Xem trước Excel và biểu đồ hội tụ
import pandas as pd
from IPython.display import Image, display
from openpyxl import load_workbook

if not duong_dan_output.is_file():
    raise FileNotFoundError("Chưa có file kết quả. Hãy chạy cell thuật toán trước.")

workbook = load_workbook(duong_dan_output, read_only=True, data_only=True)
print("Các sheet:", workbook.sheetnames)
ten_sheet = workbook.sheetnames[0]
workbook.close()

display(pd.read_excel(duong_dan_output, sheet_name=ten_sheet).head(20))

ma_phuong_phap = "ga_repair_sls" if DUNG_SOFT_LOCAL_SEARCH else "ga_repair"
duong_dan_bieu_do = thu_muc_ket_qua / f"convergence_{ma_phuong_phap}.png"
if duong_dan_bieu_do.is_file():
    display(Image(filename=str(duong_dan_bieu_do)))

In [ ]:
#@title Tải toàn bộ kết quả về máy (tùy chọn)
TAI_KET_QUA = False #@param {type:"boolean"}

if TAI_KET_QUA:
    from google.colab import files
    tep_zip = shutil.make_archive(
        "/content/ket_qua_genetic_alo",
        "zip",
        root_dir=THU_MUC_DU_AN / "outputs",
    )
    files.download(tep_zip)
else:
    print("ℹ️ Kết quả đã nằm trên Google Drive; bật TAI_KET_QUA nếu muốn tải ZIP.")